In [5]:
from google.colab import files
uploaded = files.upload()

Saving moex_close_2015_2025_no_apimoex.csv to moex_close_2015_2025_no_apimoex.csv


In [6]:
import pandas as pd

df = pd.read_csv("moex_close_2015_2025_no_apimoex.csv")
df.head()

,date,close,SECID
0,2015-01-05,NaN,ABRD
1,2015-01-06,NaN,ABRD
2,2015-01-08,99.0,ABRD
3,2015-01-09,90.0,ABRD
4,2015-01-12,90.0,ABRD


In [9]:
missing_close = (
    df
    .groupby('SECID')['close']
    .apply(lambda x: x.isna().sum())
    .sort_values(ascending=False)
)

missing_close.head(20)

,close
SECID,
PRMB,1100
NNSB,768
ABRD,309
NKSH,287
KUZB,141
BRZL,136
UKUZ,119
NKNC,23
VTBR,22


In [12]:
all_dates = df['date'].drop_duplicates().sort_values()

missing_dates = (
    df.groupby('SECID')['date']
    .apply(lambda x: len(all_dates) - x.nunique())
    .sort_values(ascending=False)
)

missing_dates.head(20)

,date
SECID,
ABRD,0
ALRS,0
APTK,0
BANE,0
BRZL,0
CHMF,0
KUZB,0
MGNT,0
MOEX,0


In [24]:
df['date'] = pd.to_datetime(df['date'])
pivot = (
    df
    .pivot(index='date', columns='SECID', values='close')
    .sort_index()
)

missing_ratio = pivot.isna().sum() / pivot.shape[0]

# Фильтрация
threshold = 0.4
valid_tickers = missing_ratio[missing_ratio <= threshold].index

pivot_clean = pivot[valid_tickers]

In [25]:
pivot_clean.head()

SECID,ABRD,ALRS,APTK,BANE,BRZL,CHMF,KUZB,MGNT,MOEX,MTLR,...,PIKK,PLZL,PRMB,RASP,SBER,SNGS,TATN,TRMK,UKUZ,VTBR
date,,,,,,,,,,,,,,,,,,,,,
2015-01-05,NaN,60.38,12.94,1245.0,499.0,522.00,NaN,9877.0,60.40,24.97,...,191.7,1000.5,NaN,23.15,56.37,24.240,238.00,37.5,598.0,0.06750
2015-01-06,NaN,61.28,13.46,1241.0,NaN,556.90,NaN,10400.0,61.90,25.30,...,198.9,1047.0,NaN,23.43,58.28,25.015,228.75,37.9,640.0,0.06666
2015-01-08,99.0,60.20,13.90,1281.0,500.0,542.70,NaN,10627.0,61.90,26.40,...,191.2,1060.0,NaN,24.25,65.70,26.265,245.00,38.0,573.0,0.06741
2015-01-09,90.0,61.91,13.69,1291.0,500.0,548.55,NaN,10542.0,60.14,26.50,...,192.7,1050.0,NaN,23.80,63.10,25.650,234.05,39.2,589.0,0.06531
2015-01-12,90.0,63.00,13.57,1270.0,431.0,558.45,NaN,10689.0,59.59,26.40,...,188.0,1020.0,NaN,23.22,62.90,25.450,228.25,38.4,588.0,0.06353


In [52]:
# Задание 2а
# Доходности
returns = pivot_clean.pct_change().iloc[1:]

# очистка выбросов
returns = returns.copy()
returns[returns > 1] = np.nan
returns[returns < -1] = np.nan

# Окна (в торговых днях)
windows = {
    "1Y": 252,
    "1Q": 63,
    "1M": 21,
    "1W": 5,
    "1D": 1
}

# Даты
target_dates = pd.to_datetime([
    "2025-12-01",
    "2022-02-16",
    "2018-06-14"
])

# Ближайшая торговая дата
def get_nearest_date(target_date, index):
    return index[index.get_indexer([target_date], method='nearest')[0]]

# Расчёт
results = {}

for date in target_dates:
    nearest_date = get_nearest_date(date, returns.index)
    results[nearest_date] = {}

    end_loc = returns.index.get_loc(nearest_date)

    for name, window in windows.items():

        if end_loc < window:
            mean_returns = None
            cov_matrix = None

        else:
            window_data = returns.iloc[end_loc - window:end_loc]

            # вектор доходностей
            mean_returns = window_data.mean()

            # ковариационная матрица
            cov_matrix = window_data.cov()

        results[nearest_date][name] = {
            "mean": mean_returns,
            "cov": cov_matrix
        }

# Вывод
for date, data in results.items():
    print(f"\n Дата: {date} ")

    for w, res in data.items():
        print(f"\n Окно: {w} ")

        if res["mean"] is not None:
            print("\n Вектор доходностей:")
            print(res["mean"].round(6))

            print("\n Ковариационная матрица:")
            print(res["cov"].round(6))
        else:
            print("Недостаточно данных")


 Дата: 2025-12-01 00:00:00 

 Окно: 1Y 

 Вектор доходностей:
SECID
ABRD   -0.000007
ALRS   -0.000495
APTK   -0.000590
BANE   -0.001071
BRZL   -0.000573
CHMF   -0.000339
KUZB    0.000536
MGNT   -0.001543
MOEX   -0.000113
MTLR    0.000019
MTSS    0.001050
MVID   -0.000197
NKNC    0.000075
NKSH   -0.000421
NLMK   -0.000273
NMTP    0.000484
NNSB    0.001697
NVTK    0.001731
PHOR    0.000650
PIKK    0.001224
PLZL   -0.001772
PRMB    0.000282
RASP   -0.001199
SBER    0.001237
SNGS   -0.000230
TATN    0.000700
TRMK    0.000641
UKUZ   -0.000431
VTBR    0.000727
dtype: float64

 Ковариационная матрица:
SECID      ABRD      ALRS      APTK      BANE      BRZL      CHMF      KUZB  \
SECID                                                                         
ABRD   0.000574  0.000177  0.000136  0.000171  0.000118  0.000187  0.000139   
ALRS   0.000177  0.000431  0.000143  0.000259  0.000109  0.000376  0.000082   
APTK   0.000136  0.000143  0.000299  0.000177  0.000086  0.000183  0.000107   
BA

/tmp/ipykernel_5752/3959729786.py:3: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = pivot_clean.pct_change().iloc[1:]
/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:11211: RuntimeWarning: Degrees of freedom <= 0 for slice
  base_cov = np.cov(mat.T, ddof=ddof)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [42]:
import numpy as np

analysis_results = []

for date, data in results.items():
    for window, res in data.items():

        cov = res["cov"]
        mean = res["mean"]

        if cov is None:
            continue

        # Средняя дисперсия (средний риск акций)
        avg_var = np.mean(np.diag(cov))

        # Общий риск системы
        total_risk = np.trace(cov)

        # Максимальный риск (самая волатильная акция)
        max_var = np.max(np.diag(cov))

        # Средняя корреляция
        std = np.sqrt(np.diag(cov))
        corr = cov / np.outer(std, std)
        avg_corr = corr.values[np.triu_indices_from(corr, k=1)].mean()

        # Средняя доходность
        avg_return = mean.mean()

        analysis_results.append({
            "date": date,
            "window": window,
            "avg_var": avg_var,
            "total_risk": total_risk,
            "max_var": max_var,
            "avg_corr": avg_corr,
            "avg_return": avg_return
        })

analysis_df = pd.DataFrame(analysis_results)

# Вывод
analysis_df.sort_values(["date", "window"])

,date,window,avg_var,total_risk,max_var,avg_corr,avg_return
14,2018-06-14,1D,NaN,NaN,NaN,NaN,-0.002578
12,2018-06-14,1M,0.000358,0.010393,0.002179,0.042915,-0.001287
11,2018-06-14,1Q,0.000631,0.018291,0.005219,0.218765,-0.000475
13,2018-06-14,1W,0.000388,0.011253,0.006015,NaN,-0.000782
10,2018-06-14,1Y,0.000498,0.014446,0.003511,0.117718,0.000612
9,2022-02-16,1D,NaN,NaN,NaN,NaN,0.024343
7,2022-02-16,1M,0.000984,0.028532,0.003844,0.487485,0.001376
6,2022-02-16,1Q,0.000766,0.022204,0.002613,0.386584,-0.001206
8,2022-02-16,1W,0.001329,0.038544,0.011371,0.508596,0.006612
5,2022-02-16,1Y,0.000829,0.024055,0.004099,0.171935,0.000876


In [ ]:
#Выводы

#1. Зависимость оценок от длины окна
#Оценки риска и взаимосвязи активов существенно зависят от длины используемого окна:
#при сокращении окна наблюдается рост волатильности и корреляций, что свидетельствует о высокой чувствительности краткосрочных оценок к рыночному шуму.

#2. Нестабильность краткосрочных оценок
#Короткие окна (1 неделя, 1 месяц) дают более волатильные и нестабильные оценки ковариации,
#что делает их менее надёжными для построения портфеля по сравнению с более длинными окнами.

#3. Кризисный эффект (2022)
#В кризисный период (2022 год) наблюдается резкий рост как волатильности, так и корреляции между активами,
#что указывает на синхронизацию их движения и существенное снижение возможностей диверсификации.

#4. Ограниченность диверсификации
#Во всех рассмотренных периодах ковариации между активами в основном положительны, что указывает на ограниченные возможности снижения риска за счёт диверсификации,
#особенно в стрессовые периоды.

#5. Различие между периодами
#В спокойные периоды (например, 2018 год) наблюдаются более низкие и менее стабильные корреляции,
#что создаёт лучшие условия для диверсификации по сравнению с более поздними периодами.

In [51]:
# Задание 2б
# Доходности
returns = pivot_clean.pct_change().iloc[1:]

# очистка выбросов
returns = returns.copy()
returns[returns > 1] = np.nan
returns[returns < -1] = np.nan

# Шаги (в торговых днях)
steps = {
    "1Y": 252,
    "1Q": 63,
    "1M": 21,
    "1W": 5,
    "1D": 1
}

# Даты
target_dates = pd.to_datetime([
    "2025-12-01",
    "2022-02-16",
    "2018-06-14"
])

# Ближайшая торговая дата
def get_nearest_date(target_date, index):
    return index[index.get_indexer([target_date], method='nearest')[0]]

# расчёт
expanding_results = {}

for date in target_dates:
    nearest_date = get_nearest_date(date, returns.index)
    expanding_results[nearest_date] = {}

    end_loc = returns.index.get_loc(nearest_date)

    for name, step in steps.items():

        if end_loc < step:
            mean_returns = None
            cov_matrix = None

        else:
            # expanding: от начала до текущей точки
            window_data = returns.iloc[:end_loc]

            # (шаг здесь влияет только на условие минимального размера окна)

            mean_returns = window_data.mean()
            cov_matrix = window_data.cov()

        expanding_results[nearest_date][name] = {
            "mean": mean_returns,
            "cov": cov_matrix
        }

# Вывод
for date, data in expanding_results.items():
    print(f"\n Дата: {date} ")

    for w, res in data.items():
        print(f"\n Окно (expanding): {w} ")

        if res["mean"] is not None:
            print("\nВектор доходностей:")
            print(res["mean"].round(6))

            print("\nКовариационная матрица:")
            print(res["cov"].round(6))
        else:
            print("Недостаточно данных")

/tmp/ipykernel_5752/1612471463.py:3: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = pivot_clean.pct_change().iloc[1:]



 Дата: 2025-12-01 00:00:00 

 Окно (expanding): 1Y 

Вектор доходностей:
SECID
ABRD    0.000611
ALRS    0.000055
APTK    0.000146
BANE    0.000286
BRZL    0.000785
CHMF    0.000411
KUZB    0.001075
MGNT   -0.000221
MOEX    0.000542
MTLR    0.000975
MTSS    0.000236
MVID    0.000034
NKNC    0.000754
NKSH    0.000966
NLMK    0.000377
NMTP    0.000987
NNSB    0.001458
NVTK    0.000563
PHOR    0.000610
PIKK    0.000562
PLZL    0.001019
PRMB    0.001212
RASP    0.001112
SBER    0.000841
SNGS    0.000190
TATN    0.000592
TRMK    0.000714
UKUZ    0.000795
VTBR   -0.000276
dtype: float64

Ковариационная матрица:
SECID      ABRD      ALRS      APTK      BANE      BRZL      CHMF      KUZB  \
SECID                                                                         
ABRD   0.000926  0.000077  0.000110  0.000077  0.000069  0.000084  0.000122   
ALRS   0.000077  0.000411  0.000084  0.000128  0.000085  0.000170  0.000069   
APTK   0.000110  0.000084  0.000664  0.000084  0.000070  0.000086  0.00

In [53]:
analysis_expanding = []

for date, data in expanding_results.items():
    for window, res in data.items():

        cov = res["cov"]
        mean = res["mean"]

        if cov is None:
            continue

        # Средняя дисперсия
        avg_var = np.mean(np.diag(cov))

        # Общий риск
        total_risk = np.trace(cov)

        # Максимальная дисперсия
        max_var = np.max(np.diag(cov))

        # Средняя корреляция
        std = np.sqrt(np.diag(cov))
        corr = cov / np.outer(std, std)
        avg_corr = corr.values[np.triu_indices_from(corr, k=1)].mean()

        # Средняя доходность
        avg_return = mean.mean()

        analysis_expanding.append({
            "date": date,
            "window": window,
            "avg_var": avg_var,
            "total_risk": total_risk,
            "max_var": max_var,
            "avg_corr": avg_corr,
            "avg_return": avg_return
        })

analysis_expanding_df = pd.DataFrame(analysis_expanding)
analysis_expanding_df.sort_values(["date", "window"])

,date,window,avg_var,total_risk,max_var,avg_corr,avg_return
14,2018-06-14,1D,0.000720,0.020867,0.002804,0.093083,0.000977
12,2018-06-14,1M,0.000720,0.020867,0.002804,0.093083,0.000977
11,2018-06-14,1Q,0.000720,0.020867,0.002804,0.093083,0.000977
13,2018-06-14,1W,0.000720,0.020867,0.002804,0.093083,0.000977
10,2018-06-14,1Y,0.000720,0.020867,0.002804,0.093083,0.000977
9,2022-02-16,1D,0.000699,0.020258,0.002191,0.120328,0.000880
7,2022-02-16,1M,0.000699,0.020258,0.002191,0.120328,0.000880
6,2022-02-16,1Q,0.000699,0.020258,0.002191,0.120328,0.000880
8,2022-02-16,1W,0.000699,0.020258,0.002191,0.120328,0.000880
5,2022-02-16,1Y,0.000699,0.020258,0.002191,0.120328,0.000880


In [ ]:
# Выводы
# 1. Независимость от длины окна
# При использовании расширяющегося окна оценки риска и доходности не зависят от выбранного горизонта (1Y, 1M, 1W и т.д.),
# так как расчёты ведутся на всей накопленной истории наблюдений.

# 2. Стабильность оценок риска
# Оценки волатильности и совокупного риска практически не изменяются во времени, что свидетельствует о высокой устойчивости метода,
# но одновременно указывает на его низкую чувствительность к текущим изменениям рынка.

# 3. Сглаживание кризисных эффектов
# В отличие от скользящего окна, расширяющееся окно не фиксирует резкий рост риска в кризисный период (2022 год),
# так как влияние шоков размывается за счёт включения всей исторической выборки.

# 4. Рост корреляций во времени
# Наблюдается постепенное увеличение средней корреляции между активами,
# что может свидетельствовать о росте синхронности движения рынка и снижении эффективности диверсификации.

# 5. Ограниченная применимость для портфельного управления
# Несмотря на стабильность, расширяюшееся окно менее пригодно для практического управления портфелем,
# так как не отражает актуальные рыночные условия и может приводить к запаздывающим инвестиционным решениям.

# 6. Ключевое отличие от скользящего
# Расширяющееся окно обеспечивает устойчивые, но инертные оценки,
# тогда как скользящее окно позволяет лучше учитывать текущую рыночную динамику и кризисные изменения.

In [56]:
# Задание 3

# Очистка выбросов
returns = returns.copy()
returns[(returns > 1) | (returns < -1)] = np.nan

# Параметр забывания
lambda_ = 0.94

# Даты
target_dates = pd.to_datetime([
    "2025-12-01",
    "2022-02-16",
    "2018-06-14"
])

# Ближайшая дата
def get_nearest_date(target_date, index):
    return index[index.get_indexer([target_date], method='nearest')[0]]

# EWMA ковариация
def ewma_covariance(data, lambda_):
    n_assets = data.shape[1]
    cov = np.zeros((n_assets, n_assets))

    # начинаем с нулевой матрицы
    for i in range(len(data)):
        r = data.iloc[i].values.reshape(-1, 1)

        if np.isnan(r).any():
            continue

        cov = lambda_ * cov + (1 - lambda_) * (r @ r.T)

    return pd.DataFrame(cov, index=data.columns, columns=data.columns)

# Расчёт
ewma_results = {}

for date in target_dates:
    nearest_date = get_nearest_date(date, returns.index)

    window_data = returns.loc[:nearest_date]

    mean_returns = window_data.mean()
    cov_matrix = ewma_covariance(window_data, lambda_)

    ewma_results[nearest_date] = {
        "mean": mean_returns,
        "cov": cov_matrix
    }

for date, res in ewma_results.items():
    print(f"\n Дата: {date}")

    print("\nВектор доходностей:")
    print(res["mean"].round(6))

    print("\nКовариационная матрица (EWMA):")
    print(res["cov"].round(6))


 Дата: 2025-12-01 00:00:00

Вектор доходностей:
SECID
ABRD    0.000609
ALRS    0.000054
APTK    0.000148
BANE    0.000285
BRZL    0.000783
CHMF    0.000413
KUZB    0.001075
MGNT   -0.000227
MOEX    0.000542
MTLR    0.000991
MTSS    0.000240
MVID    0.000033
NKNC    0.000753
NKSH    0.000975
NLMK    0.000385
NMTP    0.000984
NNSB    0.001470
NVTK    0.000568
PHOR    0.000610
PIKK    0.000564
PLZL    0.001020
PRMB    0.001217
RASP    0.001112
SBER    0.000842
SNGS    0.000190
TATN    0.000587
TRMK    0.000713
UKUZ    0.000795
VTBR   -0.000278
dtype: float64

Ковариационная матрица (EWMA):
SECID      ABRD      ALRS      APTK      BANE      BRZL      CHMF      KUZB  \
SECID                                                                         
ABRD   0.000121  0.000087  0.000068  0.000097  0.000040  0.000104  0.000024   
ALRS   0.000087  0.000204  0.000100  0.000114  0.000071  0.000231  0.000068   
APTK   0.000068  0.000100  0.000133  0.000087  0.000029  0.000129  0.000031   
BANE   0.0

In [57]:
comparison = []

for date in ewma_results.keys():

    # EWMA
    cov_ewma = ewma_results[date]["cov"]
    mean_ewma = ewma_results[date]["mean"]

    # Rolling
    cov_roll = results[date]["1Y"]["cov"]
    mean_roll = results[date]["1Y"]["mean"]

    if cov_roll is None or cov_ewma is None:
        continue

    # функции для метрик
    def compute_metrics(cov, mean):
        avg_var = np.mean(np.diag(cov))
        total_risk = np.trace(cov)
        max_var = np.max(np.diag(cov))

        std = np.sqrt(np.diag(cov))
        corr = cov / np.outer(std, std)
        avg_corr = corr.values[np.triu_indices_from(corr, k=1)].mean()

        avg_return = mean.mean()

        return avg_var, total_risk, max_var, avg_corr, avg_return

    # расчет
    roll_metrics = compute_metrics(cov_roll, mean_roll)
    ewma_metrics = compute_metrics(cov_ewma, mean_ewma)

    comparison.append({
        "date": date,

        "roll_avg_var": roll_metrics[0],
        "ewma_avg_var": ewma_metrics[0],

        "roll_total_risk": roll_metrics[1],
        "ewma_total_risk": ewma_metrics[1],

        "roll_max_var": roll_metrics[2],
        "ewma_max_var": ewma_metrics[2],

        "roll_avg_corr": roll_metrics[3],
        "ewma_avg_corr": ewma_metrics[3],

        "roll_avg_return": roll_metrics[4],
        "ewma_avg_return": ewma_metrics[4],

        # разница
        "risk_diff": ewma_metrics[1] - roll_metrics[1],
        "corr_diff": ewma_metrics[3] - roll_metrics[3]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,date,roll_avg_var,ewma_avg_var,roll_total_risk,ewma_total_risk,roll_max_var,ewma_max_var,roll_avg_corr,ewma_avg_corr,roll_avg_return,ewma_avg_return,risk_diff,corr_diff
0,2025-12-01,0.000662,0.000380,0.019206,0.011015,0.003624,0.001763,0.366808,0.475508,0.000062,0.000602,-0.008190,0.108700
1,2022-02-16,0.000829,0.000819,0.024055,0.023743,0.004099,0.003498,0.171935,0.414919,0.000876,0.000887,-0.000312,0.242984
2,2018-06-14,0.000498,0.000389,0.014446,0.011290,0.003511,0.001901,0.117718,0.122802,0.000612,0.000970,-0.003156,0.005084
